# Lab 06 · Throttle tokens per caller with token limits

Add a token limit to your model so that each caller gets 1,000 tokens per minute, then send a burst of requests,
watch the gateway answer `429` with `Retry-After`, and prove that a second key has its own allowance. This notebook is
the Python track of the [lab guide](README.md): it writes the same `tokenLimit` policy as the CLI track and runs the
same burst as [infra/trigger-token-limit.sh](infra/trigger-token-limit.sh).

**Snapshot policies → Add token limit → Burst → Inspect 429 → Second key → Restore**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Backend model `foundry-models/gpt-4.1-mini` | The model from notebook 02. Its `properties.policies` array holds the token limit. |
| `tokenLimit` policy | 1,000 tokens per minute, counted per `identity`, so every runtime key gets its own counter. |
| Runtime key `app-first-call` | The caller that runs out of tokens. |
| Runtime key `lab06-second-key` | A second caller that still has a full allowance. |

![Token limits: each caller identity gets its own counter, and the gateway answers 429 when it runs out](../assets/diagrams/token-limits.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  reuses your CLI sign-in through `AzureCliCredential`.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-4.1-mini` model and alias.

Run the cells in order, or click **Run All**. Each step is idempotent: it reuses what already exists.

In [ ]:
%pip install -q --upgrade azure-identity requests openai

### 0. Initialize variables and clients

The defaults match notebooks 01 and 02. The token limit goes on the backend model `gpt-4.1-mini` under the
`foundry-models` provider, and clients call it through the alias with the same name.

In [ ]:
import hashlib, json, os, re, subprocess, sys, time
from pathlib import Path

import requests
from azure.identity import AzureCliCredential
from openai import AuthenticationError, OpenAI, RateLimitError

location        = os.environ.get("AIGW_LOCATION", "eastus2")                  # change to swedencentral if you prefer
resource_group  = os.environ.get("AIGW_RESOURCE_GROUP", "rg-ai-gateway-labs") # change to use another resource group
gateway_name    = os.environ.get("AIGW_GATEWAY_NAME", "")                     # change to match notebook 01 (empty = derived)
provider        = "foundry-models"     # model provider from notebook 02
model           = "gpt-4.1-mini"       # backend model that gets the token limit
alias           = "gpt-4.1-mini"       # name clients send in the model field
api_key_name    = "app-first-call"     # runtime API key from notebook 02
second_key_name = "lab06-second-key"   # created and deleted by this notebook
token_limit     = {"type": "tokenLimit", "count": 1000, "period": "minute", "counterKey": ["identity"]}

lab_name   = Path(globals().get("__vsc_ipynb_file__", "06-token-limits.ipynb")).stem
lab_labels = {"lab": lab_name, "workload": "ai-gateway-labs"}   # tags for clean-up

result = subprocess.run("az account show -o json", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    sys.exit("❌ Azure CLI not installed or not signed in. Run `az login` first.")
account         = json.loads(result.stdout)
subscription_id = account["id"]
tenant_id       = account["tenantId"]
user_name       = account["user"]["name"]

suffix       = hashlib.sha1(subscription_id.encode()).hexdigest()   # stable per subscription
gateway_name = gateway_name or f"aigw-labs-{suffix[:6]}"
assert location in ("eastus2", "swedencentral"), "During the preview, AI Gateway runs in eastus2 or swedencentral."
assert re.fullmatch(r"[A-Za-z]([A-Za-z0-9-]{0,22}[A-Za-z0-9])?", gateway_name), "Use 1-24 letters, digits, or hyphens."

ARM        = "https://management.azure.com"
ARM_API    = "2025-09-01-preview"   # Microsoft.ApiManagement/aigateways
GATEWAY_ID = (f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
              f"/providers/Microsoft.ApiManagement/aigateways/{gateway_name}")

credential = AzureCliCredential()   # every client below reuses your az login
_arm_token = None

def arm(method, path, body=None, api_version=ARM_API, params=None):
    """Call Azure Resource Manager and return the JSON body (None when a GET or DELETE finds nothing)."""
    global _arm_token
    if _arm_token is None or _arm_token.expires_on - time.time() < 300:
        _arm_token = credential.get_token("https://management.azure.com/.default")
    response = requests.request(method, ARM + path, json=body, timeout=120,
                                params={"api-version": api_version, **(params or {})},
                                headers={"Authorization": f"Bearer {_arm_token.token}"})
    if response.status_code == 404 and method in ("GET", "DELETE"):
        return None
    if not response.ok:
        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text[:1000]}")
    return response.json() if response.content else {}

def wait_until_ready(path, api_version=ARM_API, label="Resource", timeout_minutes=30):
    """Poll properties.provisioningState until it reports Succeeded."""
    start = time.time()
    while True:
        resource = arm("GET", path, api_version=api_version)
        state = "NotFound" if resource is None else resource.get("properties", {}).get("provisioningState", "Succeeded")
        if state == "Succeeded":
            return resource
        if state in ("Failed", "Canceled"):
            raise RuntimeError(f"{label} provisioning {state}: {json.dumps(resource.get('properties'))[:1000]}")
        if time.time() - start > timeout_minutes * 60:
            raise TimeoutError(f"{label} is still {state} after {timeout_minutes} minutes.")
        print(f"⏳ {label}: {state} ({int(time.time() - start)} s)")
        time.sleep(20)

gateway = arm("GET", GATEWAY_ID)
if not gateway or gateway["properties"].get("provisioningState") != "Succeeded":
    sys.exit(f"❌ Gateway {gateway_name} not found in {resource_group}. Run 01-provision-gateway.ipynb first.")
gateway_url = "https://" + gateway["properties"]["frontend"]["defaultHostname"]
WS          = f"{GATEWAY_ID}/workspaces/default"   # models, aliases, tool servers, and policies live here

def get_or_create_key(name):
    """Create the runtime API key if it's missing, then read its primary value."""
    key_id = f"{GATEWAY_ID}/apiKeys/{name}"
    created = arm("GET", key_id) is None
    if created:
        arm("PUT", key_id, {"properties": {}})
    secrets = arm("POST", f"{key_id}/listSecrets")
    return secrets["primaryKey"], created

print(f"🧪 Lab:            {lab_name}")
print(f"👤 User:           {user_name}")
print(f"🏛️ Tenant:         {tenant_id}")
print(f"🔑 Subscription:   {account['name']} ({subscription_id})")
print(f"🌍 Region:         {location}")
print(f"📁 Resource group: {resource_group}")
print(f"🚪 Gateway:        {gateway_name}")
print(f"🌐 Runtime URL:    {gateway_url}")
print(f"🤖 Model:          {provider}/{model} (alias {alias})")

### 1. Check the model and get a key

The cell confirms that notebook 02 published the model and alias, then reuses the `app-first-call` key.

In [ ]:
MODEL_PATH = f"{WS}/modelProviders/{provider}/models/{model}"

if not arm("GET", MODEL_PATH):
    sys.exit(f"❌ Model {provider}/{model} not found. Run 02-first-model-call.ipynb first.")
aliases = arm("GET", f"{WS}/aliases")["value"]
if not any(a["properties"].get("aliasName") == alias for a in aliases):
    sys.exit(f"❌ Alias {alias} not found. Run 02-first-model-call.ipynb first.")
print(f"✅ Model {provider}/{model} and alias {alias} exist")

api_key, created_key = get_or_create_key(api_key_name)
print(("✅ Created" if created_key else "♻️ Reusing") + f" key {api_key_name}: {api_key[:4]}…")

### 2. Add the token limit

Policies live in the model's `properties.policies` array, and a `PATCH` replaces the whole array. So the cell reads
the array, keeps every policy that isn't a token limit, adds yours, and writes the result back. It saves the original
array so clean-up can restore it. Read-back items carry a `scope` field that you don't send back.

- `period` accepts `minute`, `hour`, or `day`.
- `counterKey` is an array with `identity`, `ipAddress`, or both. `identity` is **Caller identity** in the portal:
  each runtime key or Microsoft Entra identity gets its own counter.

In [ ]:
def without_scope(policies):
    return [{k: v for k, v in p.items() if k != "scope"} for p in policies or []]

current = without_scope(arm("GET", MODEL_PATH)["properties"].get("policies"))
original_policies = [p for p in current if p != token_limit]   # ignore a limit left over from an earlier run

if token_limit in current and len([p for p in current if p["type"] == "tokenLimit"]) == 1:
    print("♻️ Token limit already in place")
else:
    policies = [p for p in current if p["type"] != "tokenLimit"] + [token_limit]
    arm("PATCH", MODEL_PATH, {"properties": {"policies": policies}})
    print("✅ Added the token limit")

applied = arm("GET", MODEL_PATH)["properties"]["policies"]
print(json.dumps(applied, indent=2))
limit = next(p for p in applied if p["type"] == "tokenLimit")
assert limit["count"] == 1000 and limit["period"] == "minute" and limit["counterKey"] == ["identity"]

### 3. Send a burst

Send eight requests that each ask for about 250 completion tokens. With a 1,000-token allowance, the first three or
four succeed and then the gateway answers `429` with `Retry-After`. Successful responses also carry the Foundry
deployment's own `x-ratelimit-remaining-tokens` header: the backend still has capacity left while the gateway throttles
this caller, which is the point of a per-caller limit.

The client sets `max_retries=0`: the OpenAI SDK normally honors `Retry-After` and retries for you, which would hide the
`429`. A new policy can take a moment to apply, so the cell repeats the burst if no request is throttled.

In [ ]:
PROMPT = "Write a 150-word product update about an AI gateway that governs models and tools."
base_url = f"{gateway_url}/default/models/openai/v1"
client   = OpenAI(base_url=base_url, api_key=api_key, max_retries=0)

def send(client, prompt=PROMPT, max_tokens=250):
    """Send one chat request. Return the status, response headers, body text, and tokens used."""
    try:
        raw = client.chat.completions.with_raw_response.create(
            model=alias, max_tokens=max_tokens, messages=[{"role": "user", "content": prompt}])
        return raw.status_code, raw.headers, raw.text, raw.parse().usage.total_tokens
    except RateLimitError as error:
        return 429, error.response.headers, error.response.text, None

def burst(client, count=8):
    print(f"{'Request':<8} {'Status':<7} {'Retry-After':<12} {'Tokens used':<12} Backend tokens left")
    results = []
    for i in range(1, count + 1):
        status, headers, body, used = send(client)
        backend_left = headers.get("x-ratelimit-remaining-tokens", "-")
        print(f"{i:<8} {status:<7} {headers.get('retry-after', '-'):<12} {used or '-':<12} {backend_left}")
        results.append((status, headers, body))
    return results

for attempt in range(4):
    results   = burst(client)
    throttled = [r for r in results if r[0] == 429]
    if throttled:
        break
    print("⏳ No 429 yet. The policy may still be applying; trying again in 20 s.")
    time.sleep(20)

assert throttled, "Expected at least one 429."
assert any(r[1].get("retry-after") for r in throttled), "A throttled response should carry Retry-After."
print(f"\n✅ {len(throttled)} of {len(results)} requests throttled with 429")

### 4. Inspect a throttled response

Look at the status, headers, and body of one throttled call. `Retry-After` tells the client how many seconds to wait.
Well-behaved SDKs, including the OpenAI SDKs, honor it automatically when they retry. The gateway rejected the
request itself: the backend didn't receive it.

In [ ]:
status, headers, body = throttled[-1]
print(f"📨 HTTP {status}")
for name, value in headers.items():
    print(f"   {name}: {value}")
print(f"   body: {body[:400]}")

### 5. Prove the counter is per caller

Create a second runtime key and call the model while the first key is still throttled. The second key starts with a
full allowance, because **Caller identity** gives every key its own counter. A new key can take a moment to work, so
the cell retries `401` with a cheap request first.

In [ ]:
second_key, created_second = get_or_create_key(second_key_name)
print(("✅ Created" if created_second else "♻️ Reusing") + f" key {second_key_name}: {second_key[:4]}…")
second_client = OpenAI(base_url=base_url, api_key=second_key, max_retries=0)

for attempt in range(20):
    try:
        send(second_client, prompt="ping", max_tokens=5)
        break
    except AuthenticationError:
        print("⏳ The new key isn't active yet...")
        time.sleep(15)

first_status = 200
for _ in range(6):   # use up the first key's allowance again
    first_status, *_ = send(client)
    if first_status == 429:
        break
second_status, *_ = send(second_client)

print(f"🔑 {api_key_name:<18} HTTP {first_status}")
print(f"🔑 {second_key_name:<18} HTTP {second_status}")
assert first_status == 429 and second_status == 200
print("✅ Each key has its own token counter")

### 6. Clean up resources

A 1,000-token limit throttles real traffic, so the cell restores the model's original policies and deletes the second
key. It deletes `app-first-call` only if this notebook created it.

To find throttled calls in **Monitoring**, configure Application Insights first (lab 13).

In [ ]:
arm("PATCH", MODEL_PATH, {"properties": {"policies": original_policies}})
restored = without_scope(arm("GET", MODEL_PATH)["properties"].get("policies"))
assert restored == original_policies, restored
print(f"♻️ Restored {len(original_policies)} original policies on {model}")

if created_second:
    arm("DELETE", f"{GATEWAY_ID}/apiKeys/{second_key_name}")
    print(f"🗑️ Deleted key {second_key_name}")
if created_key:
    arm("DELETE", f"{GATEWAY_ID}/apiKeys/{api_key_name}")
    print(f"🗑️ Deleted key {api_key_name}")

## Troubleshooting

- **No `429` after several bursts**: check that the policy is on the model you call. The alias must point to
  `foundry-models/gpt-4.1-mini`, and `arm("GET", MODEL_PATH)["properties"]["policies"]` must list the token limit.
- **`429` on the first request**: the key was throttled by an earlier run. Wait for `Retry-After` seconds.
- **`429` from the backend instead of the gateway**: the deployment's own quota ran out, and the body isn't
  `Token limit is exceeded`. Check `x-ratelimit-remaining-tokens` and `x-ratelimit-remaining-requests` on earlier
  responses, raise the deployment's capacity, or keep the sum of per-caller limits below it.
- **Every key is throttled together**: the counter isn't per caller. Use `"counterKey": ["identity"]`.
- **`PATCH` removed other policies**: `PATCH` replaces the whole array. Restore `original_policies` from step 2.
- **The OpenAI SDK waits instead of failing**: it honors `Retry-After` and retries. Set `max_retries=0` to see the
  `429`.

## Next steps

- Cap spend in dollars per caller in [Enforce cost budgets with per-caller overrides](../07-cost-budgets/README.md).
- Go keyless in [Go keyless with Microsoft Entra ID and RBAC](../11-entra-rbac/README.md) and its notebook [11-entra-rbac.ipynb](../11-entra-rbac/11-entra-rbac.ipynb).
- Docs: [Policies in AI Gateway](https://ai.gateway.azure.com/docs/policies).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/06-token-limits/](https://azure.github.io/ai-gateway/labs/06-token-limits/).